# UCS420 Cognitive Computing — Lab Assignment 10
## NLP using Python-II
**Feature extraction from text, sentiment analysis and text generation**

**Name:** Dilpreet Singh  
**Roll number:** 1024170113

Run the notebook from top to bottom. The notebook uses Python, NLTK, scikit-learn, TextBlob, VADER and TensorFlow/Keras. In Google Colab, install any missing packages using the setup cell below.

In [ ]:
# Setup: run once if packages are missing
%pip -q install nltk scikit-learn textblob vaderSentiment tensorflow

In [ ]:
import re
import nltk
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

for resource in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    nltk.download(resource, quiet=True)

STOP_WORDS = set(stopwords.words("english"))
print("Libraries and NLTK resources are ready.")

## Q1. Paragraph preprocessing and word-frequency distribution

**Chosen topic: Artificial Intelligence**

Artificial intelligence is changing the way people work, learn, and communicate. It helps computers perform tasks such as recognizing images, understanding language, and making predictions. Students can use AI tools to understand difficult concepts and practise new skills. However, AI systems should be used responsibly because their results may contain errors or bias. I believe AI is most useful when it supports human creativity and decision-making rather than replacing human judgement.

In [ ]:
paragraph = (
    "Artificial intelligence is changing the way people work, learn, and communicate. "
    "It helps computers perform tasks such as recognizing images, understanding language, "
    "and making predictions. Students can use AI tools to understand difficult concepts "
    "and practise new skills. However, AI systems should be used responsibly because their "
    "results may contain errors or bias. I believe AI is most useful when it supports human "
    "creativity and decision-making rather than replacing human judgement."
)

# 1. Convert to lowercase and remove punctuation
clean_text = re.sub(r"[^a-z0-9\s]", "", paragraph.lower())
print("Cleaned text:\n", clean_text)

# 2. Tokenize into words and sentences
words = word_tokenize(clean_text)
sentences = sent_tokenize(paragraph)
print("\nWord tokens:", words)
print("\nSentence tokens:")
for i, sentence in enumerate(sentences, 1):
    print(i, sentence)

# 3. Compare Python split() and NLTK word_tokenize()
split_tokens = clean_text.split()
nltk_tokens = word_tokenize(clean_text)
print("\nsplit() tokens:", split_tokens)
print("word_tokenize() tokens:", nltk_tokens)
print("Same tokens?", split_tokens == nltk_tokens)
print("Note: after punctuation removal, these are usually identical for this paragraph. "
      "NLTK's tokenizer handles punctuation and contractions more intelligently on raw text.")

# 4. Remove stopwords
filtered_words = [w for w in nltk_tokens if w not in STOP_WORDS]
print("\nAfter stopword removal:", filtered_words)

# 5. Word frequency distribution excluding stopwords
freq = Counter(filtered_words)
print("\nWord frequency distribution:")
for word, count in freq.most_common():
    print(f"{word}: {count}")

## Q2. Regex extraction, stopword removal, stemming and lemmatization

In [ ]:
# 1. Extract words containing only alphabets using re.findall()
alphabet_words = re.findall(r"\b[A-Za-z]+\b", paragraph)
print("Alphabet-only words:", alphabet_words)

# 2. Remove stopwords
q2_filtered = [w.lower() for w in alphabet_words if w.lower() not in STOP_WORDS]
print("\nAfter stopword removal:", q2_filtered)

# 3. Porter stemming
porter = PorterStemmer()
stemmed = [porter.stem(w) for w in q2_filtered]
print("\nPorter stems:", stemmed)

# 4. WordNet lemmatization
lemmatizer = WordNetLemmatizer()
lemmatized = [lemmatizer.lemmatize(w) for w in q2_filtered]
print("\nWordNet lemmas (default noun POS):", lemmatized)

# 5. Compare
print("\nComparison:")
for original, stem, lemma in zip(q2_filtered, stemmed, lemmatized):
    print(f"{original:18} -> stem: {stem:14} lemma: {lemma}")

print("\nExplanation: Stemming mechanically chops or transforms word endings and may "
      "produce non-dictionary forms. Lemmatization uses vocabulary and grammatical "
      "information to return a meaningful base form. Prefer lemmatization when readable, "
      "linguistically valid words matter; prefer stemming for a fast, lightweight search index.")

## Q3. Bag of Words and TF-IDF for three short texts

The three sample texts below are short technology-related headlines/reviews. The vectorizers learn a vocabulary from all three documents. For each document, the top three TF-IDF terms are printed.

In [ ]:
texts = [
    "Artificial intelligence helps doctors analyse medical images quickly.",
    "Blockchain technology stores transactions securely across a distributed network.",
    "Artificial intelligence and blockchain are transforming modern technology."
]

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# 1. Bag of Words
count_vectorizer = CountVectorizer(stop_words="english")
bow_matrix = count_vectorizer.fit_transform(texts)
print("Vocabulary:", count_vectorizer.get_feature_names_out())
print("\nBag of Words matrix:")
print(bow_matrix.toarray())

# 2. TF-IDF
tfidf_vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf_vectorizer.fit_transform(texts)
terms = tfidf_vectorizer.get_feature_names_out()
print("\nTF-IDF matrix:")
print(np.round(tfidf_matrix.toarray(), 3))

# 3. Top 3 keywords per document
for doc_index, row in enumerate(tfidf_matrix.toarray(), 1):
    top_indices = row.argsort()[::-1][:3]
    top_terms = [(terms[i], round(row[i], 3)) for i in top_indices if row[i] > 0]
    print(f"\nText {doc_index} top TF-IDF keywords: {top_terms}")

print("\nInterpretation: A term receives a higher TF-IDF score when it is frequent "
      "in one document but relatively uncommon across the collection. Shared words "
      "such as 'technology' or 'artificial' receive less distinguishing weight.")

## Q4. Jaccard similarity and cosine similarity

**Text 1 — Artificial Intelligence:** Artificial intelligence allows machines to learn from data. It can recognize patterns, understand language, and assist people in solving complex problems. AI is used in healthcare, education, transport, and many other fields.

**Text 2 — Blockchain:** Blockchain is a distributed digital ledger that records transactions in connected blocks. It uses cryptographic methods to protect records and makes unauthorized changes difficult. Blockchain is used in digital currencies, supply chains, and secure data sharing.

In [ ]:
text_ai = (
    "Artificial intelligence allows machines to learn from data. It can recognize "
    "patterns, understand language, and assist people in solving complex problems. "
    "AI is used in healthcare, education, transport, and many other fields."
)
text_blockchain = (
    "Blockchain is a distributed digital ledger that records transactions in connected "
    "blocks. It uses cryptographic methods to protect records and makes unauthorized "
    "changes difficult. Blockchain is used in digital currencies, supply chains, and "
    "secure data sharing."
)

def preprocess_for_similarity(text):
    text = re.sub(r"[^a-z\s]", " ", text.lower())
    tokens = word_tokenize(text)
    return [w for w in tokens if w not in STOP_WORDS]

tokens_ai = preprocess_for_similarity(text_ai)
tokens_blockchain = preprocess_for_similarity(text_blockchain)
print("AI tokens:", tokens_ai)
print("\nBlockchain tokens:", tokens_blockchain)

# Jaccard similarity using sets
set_ai, set_blockchain = set(tokens_ai), set(tokens_blockchain)
intersection = set_ai & set_blockchain
union = set_ai | set_blockchain
jaccard = len(intersection) / len(union) if union else 0
print("\nCommon terms:", intersection)
print("Jaccard similarity:", round(jaccard, 4))

# Cosine similarity using TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer(stop_words="english")
matrix = tfidf.fit_transform([text_ai, text_blockchain])
cosine = cosine_similarity(matrix[0:1], matrix[1:2])[0, 0]
print("Cosine similarity:", round(float(cosine), 4))

print("\nAnalysis: Jaccard treats every unique word equally and ignores frequency. "
      "TF-IDF cosine similarity accounts for term importance and vector direction. "
      "For these two texts, both metrics indicate limited overlap because they discuss "
      "different technologies. Cosine similarity often gives more nuanced insights "
      "when document length and term importance vary; Jaccard is simpler and useful "
      "for comparing sets of unique terms.")

## Q5. Product/service reviews and sentiment analysis

The following are sample reviews written for this assignment. TextBlob and VADER are both applied to each review. The VADER compound score is used for the Positive / Negative / Neutral label.

In [ ]:
reviews = [
    "The headphones have excellent sound quality and the battery lasts all day. I love them.",
    "The delivery was late and the product stopped working after two days. Very disappointing.",
    "The design is average. It performs the basic functions and the price is reasonable.",
    "Amazing customer service! The team quickly replaced my faulty device.",
    "The app crashes frequently and the latest update made everything worse.",
    "The package arrived on time. The product is okay, nothing special."
]

# TextBlob polarity and subjectivity
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()
results = []

for review in reviews:
    blob = TextBlob(review)
    polarity = blob.sentiment.polarity
    subjectivity = blob.sentiment.subjectivity
    compound = analyzer.polarity_scores(review)["compound"]

    if compound >= 0.05:
        label = "Positive"
    elif compound <= -0.05:
        label = "Negative"
    else:
        label = "Neutral"

    results.append((review, polarity, subjectivity, compound, label))

print(f"{'Review':65} {'TB polarity':>11} {'TB subjectivity':>15} {'VADER':>8} {'Label':>10}")
for review, polarity, subjectivity, compound, label in results:
    print(f"{review[:62]:65} {polarity:11.3f} {subjectivity:15.3f} {compound:8.3f} {label:>10}")

# 3. Create a word cloud for positive reviews
positive_reviews = " ".join(review for review, _, _, _, label in results if label == "Positive")
try:
    from wordcloud import WordCloud
    cloud = WordCloud(width=900, height=450, background_color="white",
                      stopwords=STOP_WORDS).generate(positive_reviews)
    plt.figure(figsize=(12, 5))
    plt.imshow(cloud, interpolation="bilinear")
    plt.axis("off")
    plt.title("Word Cloud — Positive Reviews")
    plt.show()
except ImportError:
    print("\nTo display the word cloud, install wordcloud with: %pip install wordcloud")
    print("Positive review text:", positive_reviews)

print("\nInterpretation: TextBlob polarity ranges from -1 (negative) to +1 (positive), "
      "while subjectivity ranges from 0 (objective) to 1 (subjective). VADER's compound "
      "score ranges from -1 to +1 and is designed for sentiment in short texts. "
      "The models can disagree because they use different lexicons and scoring methods.")

## Q6. Text generation using an LSTM model

A small training corpus of approximately 100 words is used. The code tokenizes the corpus with Keras `Tokenizer`, creates next-word input sequences, trains a small LSTM model, and generates 2–3 lines beginning with a seed phrase. Results are stochastic and may vary between runs. For better language quality, a much larger corpus and more training are required.

In [ ]:
training_text = """
artificial intelligence is changing the world of technology.
machines learn patterns from data and help people solve problems.
students use computers to study science mathematics and language.
good technology can make education more useful and accessible.
people should use intelligent systems carefully and responsibly.
data helps a computer learn useful patterns from examples.
language models predict the next word in a sequence.
deep learning uses neural networks to learn complex relationships.
a recurrent network can remember information from earlier words.
an lstm network is designed to learn long term dependencies.
"""

# Tokenizer and next-word training sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

tokenizer = Tokenizer()
tokenizer.fit_on_texts([training_text])
total_words = len(tokenizer.word_index) + 1
input_sequences = []

for line in training_text.strip().split("\n"):
    sequence = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(sequence)):
        input_sequences.append(sequence[:i + 1])

max_sequence_len = max(len(seq) for seq in input_sequences)
padded = pad_sequences(input_sequences, maxlen=max_sequence_len, padding="pre")
X = padded[:, :-1]
y = padded[:, -1]

print("Vocabulary size:", total_words)
print("Number of training sequences:", len(input_sequences))
print("Input shape:", X.shape)

# Build and train LSTM
model = Sequential([
    Embedding(total_words, 32, input_length=max_sequence_len - 1),
    LSTM(64),
    Dense(total_words, activation="softmax")
])
model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
history = model.fit(X, y, epochs=100, verbose=0)
print("Training complete. Final loss:", round(float(history.history["loss"][-1]), 4))

# Generate text from a seed phrase
def generate_text(seed_text, next_words=35):
    output = seed_text
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([output])[0]
        token_list = pad_sequences([token_list], maxlen=max_sequence_len - 1, padding="pre")
        probabilities = model.predict(token_list, verbose=0)[0]
        next_index = int(np.argmax(probabilities))
        next_word = tokenizer.index_word.get(next_index, "")
        if not next_word:
            break
        output += " " + next_word
    return output

seed = "artificial intelligence"
generated = generate_text(seed, next_words=35)
print("\nGenerated text (seed:", repr(seed), "):\n")
print(generated)

# Format the generated sequence into approximately 2–3 lines
words_out = generated.split()
for start in range(0, len(words_out), 12):
    print(" ".join(words_out[start:start + 12]))

## Conclusion

In this assignment, we performed basic text preprocessing, tokenization, stopword removal, stemming and lemmatization. We represented documents using Bag of Words and TF-IDF, compared texts with Jaccard and cosine similarity, and performed sentiment analysis using TextBlob and VADER. Finally, we trained a small LSTM language model to generate text from a seed phrase.

**Note:** Q6 uses a deliberately small demonstration corpus and a compact model. Generated text is an experiment, not a reliable or fact-checked answer.